# 07 - 评估指标**学习目标：**- 深入理解 IoU 的计算和意义- 理解 Precision、Recall、F1 的关系- 理解 mAP 的计算过程- 学会绘制 PR 曲线和混淆矩阵---

## 1. IoU (Intersection over Union)**IoU** 是评估检测框质量的核心指标：```IoU = 交集面积 / 并集面积  ┌───────────┐  │  预测框    │  │     ┌─────┼─────┐  │     │ ███ │     │    ███ = 交集  └─────┼─────┘     │        │   真实框  │        └───────────┘IoU = A∩B / A∪B```| IoU 范围 | 含义 ||----------|------|| 1.0 | 完美匹配 || 0.7+ | 很好的检测 || 0.5 | 通常的"正确"阈值 || <0.5 | 通常认为是错误检测 |

In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as patchesdef compute_iou(box1, box2):    """计算两个框的 IoU"""    x1 = max(box1[0], box2[0])    y1 = max(box1[1], box2[1])    x2 = min(box1[2], box2[2])    y2 = min(box1[3], box2[3])    intersection = max(0, x2-x1) * max(0, y2-y1)    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])    union = area1 + area2 - intersection    return intersection / union if union > 0 else 0# 可视化不同 IoU 值fig, axes = plt.subplots(1, 4, figsize=(16, 4))examples = [    ([20, 20, 80, 80], [20, 20, 80, 80], "IoU = 1.0 (完美)"),    ([20, 20, 80, 80], [30, 30, 90, 90], "IoU ≈ 0.54"),    ([20, 20, 80, 80], [50, 50, 110, 110], "IoU ≈ 0.14"),    ([20, 20, 80, 80], [100, 100, 160, 160], "IoU = 0 (无重叠)"),]for ax, (b1, b2, title) in zip(axes, examples):    iou = compute_iou(b1, b2)    ax.add_patch(patches.Rectangle((b1[0], b1[1]), b1[2]-b1[0], b1[3]-b1[1],                                     fill=False, edgecolor='blue', linewidth=2, label='预测'))    ax.add_patch(patches.Rectangle((b2[0], b2[1]), b2[2]-b2[0], b2[3]-b2[1],                                     fill=False, edgecolor='red', linewidth=2, label='真实'))    ax.set_xlim(0, 180)    ax.set_ylim(0, 180)    ax.set_title(f"{title}\nIoU = {iou:.3f}")    ax.legend(fontsize=8)plt.tight_layout()plt.show()

## 2. Precision 与 Recall```                    预测结果              正例(Pred)    负例(Pred)实际  正例    TP(真正例)    FN(假负例)  ← Recall = TP/(TP+FN)      负例    FP(假正例)    TN(真负例)Precision = TP / (TP + FP)  → 预测为正的里面有多少是对的Recall    = TP / (TP + FN)  → 实际为正的里面找到了多少```**Trade-off**：- 高置信度阈值 → Precision 高, Recall 低（只报最确定的）- 低置信度阈值 → Precision 低, Recall 高（尽量不漏）

In [ ]:
# Precision-Recall Trade-off 可视化from ultralytics import YOLOfrom pathlib import Pathmodel = YOLO("yolo11n.pt")img_dir = Path("../data/coco128/images/train2017")test_imgs = [str(p) for p in sorted(img_dir.glob("*.jpg"))[:20]]thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]detection_counts = []for conf in thresholds:    results = model(test_imgs, conf=conf, verbose=False)    total = sum(len(r.boxes) for r in results)    detection_counts.append(total)plt.figure(figsize=(8, 5))plt.plot(thresholds, detection_counts, 'bo-')plt.xlabel("置信度阈值")plt.ylabel("检测数量")plt.title("置信度阈值 vs 检测数量")plt.grid(True, alpha=0.3)plt.show()

## 3. mAP (mean Average Precision)**计算步骤**：```1. 对每个类别：   a. 按置信度排序所有预测   b. 对每个预测，判断 TP/FP（IoU > 阈值？）   c. 计算累积的 Precision 和 Recall   d. 画 PR 曲线   e. AP = PR 曲线下面积2. mAP = 所有类别 AP 的平均值```**两种 mAP**：| 指标 | IoU 阈值 | 说明 ||------|----------|------|| mAP@0.5 | 固定 0.5 | 宽松，Pascal VOC 标准 || mAP@0.5:0.95 | 0.5 到 0.95 步长 0.05 | 严格，COCO 标准 |

In [ ]:
# 手动计算 AP（帮助理解原理）import numpy as npdef compute_ap_manual(recalls, precisions):    """手动计算 AP（所有点插值法）"""    # 添加哨兵值    recalls = np.concatenate(([0], recalls, [1]))    precisions = np.concatenate(([0], precisions, [0]))        # 确保 precision 单调递减    for i in range(len(precisions)-2, -1, -1):        precisions[i] = max(precisions[i], precisions[i+1])        # 找 recall 变化的点    indices = np.where(recalls[1:] != recalls[:-1])[0]        # 计算面积    ap = np.sum((recalls[indices+1] - recalls[indices]) * precisions[indices+1])    return ap# 示例：模拟一组检测结果np.random.seed(42)n_detections = 20confidences = np.random.uniform(0.3, 0.99, n_detections)is_tp = np.random.choice([True, False], n_detections, p=[0.6, 0.4])# 按置信度排序order = np.argsort(-confidences)is_tp_sorted = is_tp[order]# 计算累积 precision 和 recalltp_cumsum = np.cumsum(is_tp_sorted)fp_cumsum = np.cumsum(~is_tp_sorted)precisions = tp_cumsum / (tp_cumsum + fp_cumsum)recalls = tp_cumsum / is_tp.sum()ap = compute_ap_manual(recalls, precisions)print(f"检测数: {n_detections}, 真正例: {is_tp.sum()}")print(f"计算得到的 AP = {ap:.3f}")# 画 PR 曲线plt.figure(figsize=(8, 5))plt.plot(recalls, precisions, 'b-o', markersize=4)plt.fill_between(recalls, precisions, alpha=0.2)plt.xlabel("Recall")plt.ylabel("Precision")plt.title(f"PR Curve (AP = {ap:.3f})")plt.xlim(0, 1)plt.ylim(0, 1)plt.grid(True, alpha=0.3)plt.show()

## 4. 使用 ultralytics 评估模型

In [ ]:
from ultralytics import YOLO# 加载训练好的模型（或用预训练模型）model = YOLO("yolo11n.pt")# 在 COCO128 上评估metrics = model.val(    data="../configs/data/coco128.yaml",    device="mps",)print("=" * 50)print("评估结果：")print(f"  mAP@0.5    : {metrics.box.map50:.4f}")print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")print(f"  Precision  : {metrics.box.mp:.4f}")print(f"  Recall     : {metrics.box.mr:.4f}")print("=" * 50)

In [ ]:
# 查看每个类别的 APprint("各类别 AP (前 15 个)：")print(f"{'类别':<20} {'AP':<10}")print("-" * 30)for i, (name, ap) in enumerate(zip(model.names.values(), metrics.box.ap50)):    if i >= 15:        break    print(f"{name:<20} {ap:.4f}")

## 5. 混淆矩阵

In [ ]:
# ultralytics 可以生成混淆矩阵# 在 val 时加上 plot=Truemodel.val(    data="../configs/data/coco128.yaml",    device="mps",    plots=True,  # 生成混淆矩阵等图表)# 混淆矩阵图片保存在 runs/val/ 目录下print("混淆矩阵已保存到 val 目录")

## 6. 核心概念总结| 指标 | 公式 | 含义 ||------|------|------|| **IoU** | 交集/并集 | 框的重叠程度 || **Precision** | TP/(TP+FP) | 预测正确的比例 || **Recall** | TP/(TP+FN) | 找到的目标比例 || **AP** | PR 曲线下面积 | 单类别的综合性能 || **mAP** | 所有类别 AP 均值 | 整体性能 |---**上一节**: [06 - 迁移学习](06_transfer_learning.ipynb)**下一节**: [08 - 数据增强](08_data_augmentation.ipynb) - 数据增强策略